# Chương 5 – Phát hiện biên và phân vùng ảnh
## Bài tập thực hành

Mục tiêu:
- Cài đặt **nhân chập** với đệm 0 và các **mặt nạ phát hiện biên** (Prewitt, Sobel, Laplace) rồi kiểm tra lại với bài tập tính tay.
- Áp dụng gradient, Laplace/zero-crossing và **Canny** để phát hiện biên trên ảnh thật; khảo sát ảnh hưởng của **nhiễu** và của **làm trơn Gaussian**; ứng dụng phát hiện **làn đường**.
- Thực hành **phân vùng theo ngưỡng**: làm trơn lược đồ, **Otsu**, **ISODATA (đẳng điệu)**, **đối xứng nền**, **tam giác**, Otsu nhiều ngưỡng.
- Thực hành **phân vùng theo miền**: tách cây tứ phân, tăng trưởng vùng; mở rộng với phân cụm K-means.
- Bài tổng hợp: đếm hạt gạo và thống kê diện tích.

Các ô code có dấu `#### YOUR CODE HERE ####` là phần bạn cần hoàn thành. Hàm `check(...)` báo kết quả đúng/sai so với đáp án bài tập tính tay.

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

IMG_DIR = 'images'

def check(name, cond):
    """In kết quả kiểm tra: cond là biểu thức hoặc hàm không tham số trả về True/False."""
    try:
        ok = bool(cond() if callable(cond) else cond)
    except Exception:
        ok = False
    print(('✔ Đúng: ' if ok else '✘ Chưa đúng: ') + name)

def show(images, titles, cmap='gray', figsize=None):
    n = len(images)
    plt.figure(figsize=figsize or (5 * n, 5))
    for i, (im, t) in enumerate(zip(images, titles)):
        plt.subplot(1, n, i + 1)
        plt.imshow(im, cmap=cmap)
        plt.title(t)
        plt.axis('off')
    plt.show()

# PHẦN A. PHÁT HIỆN BIÊN

## 1. Nhân chập với đệm 0

Quy ước (giống bài tập tính tay): đặt tâm mặt nạ lên từng điểm ảnh, **nhân từng phần tử rồi cộng lại** (không lật mặt nạ); điểm nằm ngoài ảnh có giá trị 0; kết quả có cùng kích thước với ảnh.

**Yêu cầu**: viết hàm `conv2d_zero(I, H)` (có thể dùng vòng lặp hoặc `np.pad`) và kiểm tra với ảnh 7×7 của **Câu 5.T1** (khối 3×3 trên nền 4) với mặt nạ Prewitt `Hx`.

In [ ]:
I1 = np.array([[4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4], [4, 4, 18, 18, 18, 4, 4], [4, 4, 17, 14, 19, 4, 4], [4, 4, 18, 18, 18, 4, 4], [4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4]])
Hx_prewitt = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
Hy_prewitt = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])

def conv2d_zero(I, H):
    k = H.shape[0] // 2
    out = np.zeros(I.shape, dtype=np.float64)
    padded = np.pad(I.astype(np.float64), k, mode='constant')
    kh, kw = H.shape
    for i in range(I.shape[0]):
        for j in range(I.shape[1]):
            window = padded[i:i + kh, j:j + kw]
            out[i, j] = np.sum(window * H)
    return out

Ix = conv2d_zero(I1, Hx_prewitt)
Iy = conv2d_zero(I1, Hy_prewitt)
print('I ⊗ Hx =\n', Ix.astype(int))
check('I ⊗ Hx (Câu 5.T1a)', lambda: np.array_equal(Ix, np.array([[8, 0, 0, 0, 0, 0, -8], [12, 14, 14, 0, -14, -14, -12], [12, 27, 24, 2, -24, -29, -12], [12, 41, 38, 2, -38, -43, -12], [12, 27, 24, 2, -24, -29, -12], [12, 14, 14, 0, -14, -14, -12], [8, 0, 0, 0, 0, 0, -8]])))
check('I ⊗ Hy (Câu 5.T1a)', lambda: np.array_equal(Iy, np.array([[8, 12, 12, 12, 12, 12, 8], [0, 14, 28, 42, 28, 14, 0], [0, 13, 23, 38, 25, 15, 0], [0, 0, 0, 0, 0, 0, 0], [0, -13, -23, -38, -25, -15, 0], [0, -14, -28, -42, -28, -14, 0], [-8, -12, -12, -12, -12, -12, -8]])))
check('Khớp cv2.filter2D (đệm 0)', lambda: np.allclose(Ix, cv2.filter2D(I1.astype(np.float64), -1, Hx_prewitt.astype(np.float64), borderType=cv2.BORDER_CONSTANT)))

## 2. Phát hiện biên bằng gradient và Laplace trên ma trận nhỏ

### 2.1. Độ lớn gradient `|Gx| + |Gy|`

**Yêu cầu**: viết hàm `gradient_magnitude(I, Hx, Hy)` trả về `|I ⊗ Hx| + |I ⊗ Hy|`. Kiểm tra với
- Câu 5.T1: ảnh `I1` với mặt nạ **Prewitt** (ảnh `I₃`);
- Câu 5.T2: ảnh `I2` (khối 3×3 gần góc trên phải trên nền 3) với mặt nạ **Sobel**, `Hx = [[1,0,−1],[2,0,−2],[1,0,−1]]`, `Hy = [[1,2,1],[0,0,0],[−1,−2,−1]]`.

Sau đó phân ngưỡng để được ảnh biên nhị phân. Lưu ý: các hàng/cột ngoài cùng của ảnh có đáp ứng “giả” do đệm 0 (nền khác 0).

In [ ]:
I2 = np.array([[3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 12, 13, 12, 3], [3, 3, 3, 13, 11, 12, 3], [3, 3, 3, 12, 12, 13, 3], [3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3]])
Hx_sobel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
Hy_sobel = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]])

def gradient_magnitude(I, Hx, Hy):
    Gx = conv2d_zero(I, Hx)
    Gy = conv2d_zero(I, Hy)
    G = np.abs(Gx) + np.abs(Gy)
    return G

G1 = gradient_magnitude(I1, Hx_prewitt, Hy_prewitt)
G2 = gradient_magnitude(I2, Hx_sobel, Hy_sobel)
check('I₃ của Câu 5.T1 (Prewitt)', lambda: np.array_equal(G1, np.array([[16, 12, 12, 12, 12, 12, 16], [12, 28, 42, 42, 42, 28, 12], [12, 40, 47, 40, 49, 44, 12], [12, 41, 38, 2, 38, 43, 12], [12, 40, 47, 40, 49, 44, 12], [12, 28, 42, 42, 42, 28, 12], [16, 12, 12, 12, 12, 12, 16]])))
check('I₁ của Câu 5.T2 (Sobel)', lambda: np.array_equal(G2, np.array([[18, 12, 30, 50, 50, 50, 36], [12, 0, 38, 56, 36, 54, 48], [12, 0, 38, 36, 2, 36, 50], [12, 0, 38, 54, 36, 52, 50], [12, 0, 18, 36, 38, 38, 32], [12, 0, 0, 0, 0, 0, 12], [18, 12, 12, 12, 12, 12, 18]])))
print('Ảnh biên nhị phân (Câu 5.T1, ngưỡng 20):\n', (G1 >= 20).astype(int))
print('Ảnh biên nhị phân (Câu 5.T2, ngưỡng 22):\n', (G2 >= 22).astype(int))

### 2.2. Mặt nạ Laplace và điểm cắt không (zero-crossing)

Mặt nạ Laplace 8 lân cận `Hz8 = [[−1,−1,−1],[−1,8,−1],[−1,−1,−1]]` (Câu 5.T1b) và 4 lân cận `Hz4 = [[0,−1,0],[−1,4,−1],[0,−1,0]]` (Câu 5.T2b). Kết quả chứa cả giá trị âm và dương nên **chưa phải ảnh biên**.

**Yêu cầu**:
1. Tính `I₂ = I ⊗ Hz` cho hai ảnh và kiểm tra với đáp án.
2. Viết hàm `zero_crossing(Lap, thr)` đánh dấu (giá trị 1) các điểm mà **dấu thay đổi** so với điểm bên phải hoặc điểm bên dưới và hiệu `|Lap[p] − Lap[q]|` lớn hơn `thr` — các điểm cắt không là ứng viên điểm biên.

In [ ]:
Hz8 = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]])
Hz4 = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]])

Lap1 = conv2d_zero(I1, Hz8)
Lap2 = conv2d_zero(I2, Hz4)
check('I₂ của Câu 5.T1b (Hz 8 lân cận)', lambda: np.array_equal(Lap1, np.array([[20, 12, 12, 12, 12, 12, 20], [12, -14, -28, -42, -28, -14, 12], [12, -27, 75, 46, 73, -29, 12], [12, -41, 38, -32, 54, -43, 12], [12, -27, 75, 46, 73, -29, 12], [12, -14, -28, -42, -28, -14, 12], [20, 12, 12, 12, 12, 12, 20]])))
check('I₂ của Câu 5.T2b (Hz 4 lân cận)', lambda: np.array_equal(Lap2, np.array([[6, 3, 3, -6, -7, -6, 6], [3, 0, -9, 16, 14, 17, -6], [3, 0, -10, 14, -6, 9, -6], [3, 0, -9, 17, 9, 22, -7], [3, 0, 0, -9, -9, -10, 3], [3, 0, 0, 0, 0, 0, 3], [6, 3, 3, 3, 3, 3, 6]])))

def zero_crossing(Lap, thr=0):
    zc = np.zeros(Lap.shape, dtype=np.uint8)
    h, w = Lap.shape
    for i in range(h):
        for j in range(w):
            p = Lap[i, j]
            for di, dj in ((0, 1), (1, 0)):
                ni, nj = i + di, j + dj
                if ni < h and nj < w:
                    q = Lap[ni, nj]
                    if p * q < 0 and abs(p - q) > thr:
                        zc[i, j] = 1
                        break
    return zc

zc1 = zero_crossing(Lap1, 30)
print('Điểm cắt không của ảnh T1:\n', zc1)
check('Có điểm cắt không dọc theo mép khối (ít nhất 8 điểm)', lambda: zc1.sum() >= 8)

**Trả lời**: `I ⊗ Hx` và `I ⊗ Hy` có dấu biểu diễn hướng biến thiên. Nếu cộng trực tiếp, hai đáp ứng trái dấu có thể triệt tiêu nhau, nên cần tính độ lớn gradient như `|Gx| + |Gy|` (hoặc `sqrt(Gx² + Gy²)`) rồi phân ngưỡng. Ảnh Laplace cũng chứa cả giá trị âm và dương; biên nằm tại vùng đổi dấu, vì vậy cần tìm zero-crossing và thường phân ngưỡng độ chênh để loại đáp ứng yếu/nhiễu.


## 3. Phát hiện biên trên ảnh thật: Sobel, Prewitt, Canny

Với ảnh thật cần **làm trơn** (Gaussian) trước để giảm nhiễu rồi mới tính gradient. Quy trình chung: đọc ảnh xám → `cv2.GaussianBlur` → đạo hàm theo x và y → `|Gx| + |Gy|` → loại các điểm có cường độ nhỏ hơn ngưỡng `thr`.

**Yêu cầu**: hoàn thành ba hàm `sobel_edge`, `prewitt_edge`, `canny_edge` (đầu vào là ảnh xám, đầu ra là ảnh biên nhị phân 0/255).

### 3.1. Sobel – dùng `cv2.Sobel(..., cv2.CV_64F, ...)` (dùng `CV_64F` để không mất giá trị âm)

In [ ]:
def sobel_edge(gray, blur_ksize=5, thr=60):
    smoothed = gray if blur_ksize <= 1 else cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)
    gx = cv2.Sobel(smoothed, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(smoothed, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.abs(gx) + np.abs(gy)
    edge = np.where(magnitude >= thr, 255, 0).astype(np.uint8)
    return edge

### 3.2. Prewitt – dùng `cv2.filter2D` với hai mặt nạ Prewitt (đã khai báo ở Mục 1)

In [ ]:
def prewitt_edge(gray, blur_ksize=5, thr=45):
    smoothed = gray if blur_ksize <= 1 else cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)
    gx = cv2.filter2D(smoothed.astype(np.float64), -1, Hx_prewitt.astype(np.float64))
    gy = cv2.filter2D(smoothed.astype(np.float64), -1, Hy_prewitt.astype(np.float64))
    magnitude = np.abs(gx) + np.abs(gy)
    edge = np.where(magnitude >= thr, 255, 0).astype(np.uint8)
    return edge

### 3.3. Canny – dùng `cv2.Canny(ảnh_đã_làm_trơn, t1, t2)`

`t1`, `t2` là ngưỡng thấp và ngưỡng cao của bước phân ngưỡng kép (hysteresis).

In [ ]:
def canny_edge(gray, blur_ksize=5, t1=50, t2=150):
    smoothed = gray if blur_ksize <= 1 else cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)
    edge = cv2.Canny(smoothed, t1, t2)
    return edge

seed = cv2.imread(os.path.join(IMG_DIR, 'seed.png'), cv2.IMREAD_GRAYSCALE)
rice = cv2.imread(os.path.join(IMG_DIR, 'gao3.png'), cv2.IMREAD_GRAYSCALE)
for name, im in [('seed.png', seed), ('gao3.png', rice)]:
    show([im, sobel_edge(im), prewitt_edge(im), canny_edge(im)], [name, 'Sobel', 'Prewitt', 'Canny'], figsize=(20, 5))

### 3.4. So sánh `√(Gx² + Gy²)` và `|Gx| + |Gy|`

**Yêu cầu**: với ảnh `seed.png` sau khi làm trơn, tính độ lớn gradient Sobel bằng hai công thức; hiển thị hai ảnh độ lớn và tính tỉ lệ `|Gx|+|Gy|` / `√(Gx²+Gy²)` tại các điểm có `√(Gx²+Gy²) > 50`. Giá trị này nằm trong khoảng nào? Vì sao?

In [ ]:
g = cv2.GaussianBlur(seed, (5, 5), 0)
gx = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)
mag_sqrt = np.sqrt(gx ** 2 + gy ** 2)
mag_abs = np.abs(gx) + np.abs(gy)
valid = mag_sqrt > 50
ratio = mag_abs[valid] / mag_sqrt[valid]
print(f'Tỉ lệ |Gx|+|Gy| / sqrt(Gx²+Gy²): nhỏ nhất = {ratio.min():.3f}, lớn nhất = {ratio.max():.3f}, trung bình = {ratio.mean():.3f}')
check('Tỉ lệ nằm trong [1, √2]', lambda: ratio.min() >= 1 - 1e-9 and ratio.max() <= np.sqrt(2) + 1e-9)
show([mag_sqrt, mag_abs], ['√(Gx² + Gy²)', '|Gx| + |Gy|'], figsize=(10, 5))

**Trả lời**: Tỉ lệ nằm trong `[1, sqrt(2)]`. Đây là quan hệ giữa chuẩn L1 và chuẩn L2 của vectơ `(Gx, Gy)`: hai công thức bằng nhau khi một thành phần bằng 0, còn tỉ lệ đạt `sqrt(2)` khi `|Gx| = |Gy|`.


## 4. Ảnh hưởng của nhiễu và của làm trơn Gaussian

Thư mục `images/objects` có ảnh `img3.png` (sạch) và `img3_bruit.png`, `img3_bruit2.png` (cùng cảnh nhưng có nhiễu). Đạo hàm là phép toán khuếch đại thành phần tần số cao nên rất nhạy với nhiễu.

**Yêu cầu**: viết hàm `false_edge_ratio(clean_edges, noisy_edges)` = tỉ lệ điểm biên của ảnh nhiễu **không nằm trong lân cận 2 điểm ảnh** của biên ảnh sạch (dùng `cv2.dilate` với kernel 5×5). Tính tỉ lệ này cho Sobel (`blur_ksize` = 1, tức không làm trơn, và `blur_ksize` = 7) trên cả hai ảnh nhiễu, so sánh với `canny_edge`. Nhận xét tác dụng của làm trơn. (Với `blur_ksize = 1`, `cv2.GaussianBlur` không làm gì.)

In [ ]:
def false_edge_ratio(clean_edges, noisy_edges):
    clean_neighborhood = cv2.dilate((clean_edges > 0).astype(np.uint8),
                                    np.ones((5, 5), dtype=np.uint8)) > 0
    noisy_mask = noisy_edges > 0
    total_noisy = int(noisy_mask.sum())
    false_edges = noisy_mask & ~clean_neighborhood
    ratio = 0.0 if total_noisy == 0 else float(false_edges.sum()) / total_noisy
    return ratio

clean = cv2.imread(os.path.join(IMG_DIR, 'objects', 'img3.png'), cv2.IMREAD_GRAYSCALE)
rows = []
for name in ['img3_bruit.png', 'img3_bruit2.png']:
    noisy = cv2.imread(os.path.join(IMG_DIR, 'objects', name), cv2.IMREAD_GRAYSCALE)
    for label, fn in [('Sobel, không làm trơn', lambda im: sobel_edge(im, blur_ksize=1, thr=120)),
                      ('Sobel, Gaussian 7×7', lambda im: sobel_edge(im, blur_ksize=7, thr=120)),
                      ('Canny, Gaussian 7×7', lambda im: canny_edge(im, blur_ksize=7, t1=50, t2=150))]:
        r = false_edge_ratio(fn(clean), fn(noisy))
        rows.append((name, label, r))
        print(f'{name:16s} | {label:24s} | tỉ lệ biên giả = {r:.3f}')
check('Làm trơn Gaussian làm giảm biên giả của Sobel (cả 2 ảnh nhiễu)',
      lambda: rows[1][2] < rows[0][2] and rows[4][2] < rows[3][2])
noisy = cv2.imread(os.path.join(IMG_DIR, 'objects', 'img3_bruit2.png'), cv2.IMREAD_GRAYSCALE)
show([noisy, sobel_edge(noisy, 1, 120), sobel_edge(noisy, 7, 120), canny_edge(noisy, 7)], ['Ảnh nhiễu', 'Sobel không làm trơn', 'Sobel + Gaussian 7×7', 'Canny'], figsize=(20, 5))

**Nhận xét**: Không làm trơn, nhiễu tạo nhiều biến thiên cường độ cao nên Sobel sinh nhiều biên giả. Gaussian 7×7 triệt bớt thành phần cao tần và làm giảm rõ tỉ lệ biên giả, nhưng có thể làm mờ hoặc mất biên nhỏ. Canny kết hợp làm trơn, cực đại cục bộ và ngưỡng kép nên thường cho biên mảnh, liên tục và ít nhiễu hơn.


## 5. Ảnh hưởng của hai ngưỡng trong Canny

**Yêu cầu**: với ảnh `img3.png`, thử `(t1, t2)` lần lượt là `(20, 60)`, `(50, 150)`, `(100, 200)`, `(150, 250)` (làm trơn Gaussian 5×5) và hiển thị 4 kết quả cùng số điểm biên. Nhận xét: ngưỡng tăng thì điều gì xảy ra?

In [ ]:
img3 = cv2.imread(os.path.join(IMG_DIR, 'objects', 'img3.png'), cv2.IMREAD_GRAYSCALE)
params = [(20, 60), (50, 150), (100, 200), (150, 250)]
results = [canny_edge(img3, blur_ksize=5, t1=t1, t2=t2) for t1, t2 in params]
counts = [int((r > 0).sum()) for r in results]
check('Số điểm biên giảm dần khi tăng ngưỡng', lambda: all(counts[i] > counts[i + 1] for i in range(3)))
show(results, [f'Canny {p}: {c} điểm' for p, c in zip(params, counts)], figsize=(22, 6))

## 6. Ứng dụng: phát hiện làn đường

**Yêu cầu**: với các ảnh `images/lane/lane1.png`, `lane2.png`, `lane3.png`, viết hàm `detect_lanes(img_bgr)` theo các bước:

1. Chuyển ảnh xám, làm trơn Gaussian 5×5, Canny `(50, 150)`.
2. Chỉ giữ vùng quan tâm: **nửa dưới của ảnh** (từ hàng `0.55·h` trở xuống), phần còn lại gán 0.
3. Dùng `cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=60, maxLineGap=20)` để tìm các đoạn thẳng.
4. Vẽ các đoạn tìm được lên bản sao của ảnh gốc (màu đỏ, độ dày 3).

Hàm trả về `(edges_roi, kết_quả)`.

In [ ]:
def detect_lanes(img_bgr):
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    smoothed = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(smoothed, 50, 150)
    edges_roi = np.zeros_like(edges)
    start_row = int(0.55 * h)
    edges_roi[start_row:, :] = edges[start_row:, :]
    lines = cv2.HoughLinesP(edges_roi, 1, np.pi / 180, threshold=50,
                            minLineLength=60, maxLineGap=20)
    result = img_bgr.copy()
    if lines is not None:
        for x1, y1, x2, y2 in lines.reshape(-1, 4):
            cv2.line(result, (x1, y1), (x2, y2), (0, 0, 255), 3)
    return edges_roi, result

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
for i in range(3):
    im = cv2.imread(os.path.join(IMG_DIR, 'lane', f'lane{i + 1}.png'))
    e, r = detect_lanes(im)
    axes[i, 0].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); axes[i, 0].set_title(f'lane{i + 1} – gốc')
    axes[i, 1].imshow(e, cmap='gray'); axes[i, 1].set_title('Canny + vùng quan tâm')
    axes[i, 2].imshow(cv2.cvtColor(r, cv2.COLOR_BGR2RGB)); axes[i, 2].set_title('Đoạn thẳng (Hough)')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.show()

**Nhận xét**: `lane1` dễ nhất vì hai vạch thẳng, sáng và tương phản cao. `lane2` khó vì vạch cong trong khi HoughLinesP chỉ mô hình hóa các đoạn thẳng. `lane3` khó nhất do sàn phản chiếu, nhiều cạnh của tường/đồ vật và vạch không đối xứng như làn đường thật. Có thể cải thiện bằng ROI hình thang, lọc đoạn theo độ dốc/vị trí, ghép các đoạn đồng hướng; với đường cong nên dùng biến đổi phối cảnh và khớp đa thức thay cho chỉ dùng đường thẳng Hough.


# PHẦN B. PHÂN VÙNG ẢNH

## 7. Lược đồ xám và làm trơn lược đồ

Công thức làm trơn với cửa sổ `W` (lẻ): `hsmooth(g) = (1/W) · Σ h(g − w)`, `w` chạy từ `−(W−1)/2` đến `(W−1)/2`; giá trị `h(g)` ngoài khoảng `[0, L−1]` coi bằng 0.

**Yêu cầu**: viết hàm `smooth_hist(h, W)` và kiểm tra với ảnh 6×6 của **Câu 5.T6** (10 mức xám 0–9) với `W = 3` và `W = 5`. Viết thêm hàm `count_peaks(h)` đếm số cực đại cục bộ (điểm lớn hơn điểm bên trái và lớn hơn hoặc bằng điểm bên phải, giá trị > 0), rồi áp dụng cho lược đồ 256 mức của ảnh `gao3.png` trước và sau khi làm trơn với `W = 9`.

In [ ]:
Is = np.array([[3, 4, 1, 8, 3, 3], [8, 6, 1, 8, 1, 2], [4, 8, 3, 1, 8, 5], [3, 9, 6, 8, 2, 6], [2, 0, 3, 1, 0, 7], [3, 9, 1, 8, 8, 9]])
hist6 = np.bincount(Is.ravel(), minlength=10)
print('h(g) =', hist6)

def smooth_hist(h, W):
    if W <= 0 or W % 2 == 0:
        raise ValueError('W phải là số lẻ dương')
    h = np.asarray(h, dtype=np.float64)
    hs = np.convolve(h, np.ones(W, dtype=np.float64) / W, mode='same')
    return hs

def count_peaks(h):
    h = np.asarray(h)
    return int(sum(1 for i in range(len(h))
                   if h[i] > 0 and (i == 0 or h[i] > h[i - 1]) and (i == len(h) - 1 or h[i] >= h[i + 1])))

print('W = 3:', np.round(smooth_hist(hist6, 3), 2))
print('W = 5:', np.round(smooth_hist(hist6, 5), 2))
check('hsmooth, W = 3 (Câu 5.T6)', lambda: np.allclose(np.round(smooth_hist(hist6, 3), 2), [2.67, 3.67, 5.33, 4.0, 3.33, 2.0, 1.67, 4.0, 4.0, 3.67]))
check('hsmooth, W = 5 (Câu 5.T6)', lambda: np.allclose(np.round(smooth_hist(hist6, 5), 2), [2.2, 3.6, 4.0, 3.8, 3.2, 2.8, 3.0, 3.2, 3.0, 2.4]))
check('Số đỉnh Câu 5.T6: gốc 4, W = 3 còn 2', lambda: count_peaks(hist6) == 4 and count_peaks(smooth_hist(hist6, 3)) == 2)

rice = cv2.imread(os.path.join(IMG_DIR, 'gao3.png'), cv2.IMREAD_GRAYSCALE)
h_rice = np.bincount(rice.ravel(), minlength=256)
h_rice_s = smooth_hist(h_rice, 9)
print('Ảnh gao3: số đỉnh trước làm trơn =', count_peaks(h_rice), ', sau làm trơn W = 9 =', count_peaks(h_rice_s))
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.bar(range(256), h_rice, width=1.0); plt.yscale('log'); plt.title('Lược đồ gốc (trục log)')
plt.subplot(1, 2, 2); plt.bar(range(256), h_rice_s, width=1.0); plt.yscale('log'); plt.title('Lược đồ sau làm trơn W = 9')
plt.show()

## 8. Thuật toán Otsu

Với lược đồ chuẩn hóa `p_i`, tổng tích lũy `P₁(k)`, trung bình tích lũy `m(k)` và trung bình toàn cục `m_G`:

`σ_B²(k) = [m_G · P₁(k) − m(k)]² / (P₁(k) · (1 − P₁(k)))`, ngưỡng `k*` làm cực đại `σ_B²(k)`; nếu có nhiều `k` cùng cực đại thì lấy trung bình các `k` đó. Nếu `P₁(k) = 0` hoặc `1` thì coi `σ_B²(k) = 0`.

**Yêu cầu**: viết hàm `otsu_threshold(h)` trả về `(k*, sigma)` với `sigma` là mảng `σ_B²(k)`. Kiểm tra với ảnh 6×8 của **Câu 5.T4a** (đáp án `k* = 4`) và với lược đồ nhiều ngưỡng của **Câu 5.T4b** (`k* = 4`, lấy trung bình của 3 và 5). Cuối cùng so sánh với `cv2.threshold(..., THRESH_OTSU)` trên ảnh `gao3.png`.

In [ ]:
Img = np.array([[3, 7, 3, 2, 5, 2, 3, 7], [1, 1, 2, 4, 8, 4, 1, 9], [9, 2, 7, 2, 7, 2, 2, 5], [0, 8, 7, 8, 1, 0, 2, 6], [8, 4, 5, 1, 1, 1, 6, 4], [7, 8, 2, 2, 7, 1, 1, 2]])
h_t4 = np.bincount(Img.ravel(), minlength=10)
h_tie = np.array([7, 2, 2, 1, 11, 11, 1, 2, 2, 7])

def otsu_threshold(h):
    h = np.asarray(h, dtype=np.float64)
    p = h / h.sum()
    levels = np.arange(len(h))
    P1 = np.cumsum(p)
    m = np.cumsum(levels * p)
    mG = m[-1]
    sigma = np.zeros_like(p)
    valid = (P1 > 0) & (P1 < 1)
    sigma[valid] = (mG * P1[valid] - m[valid]) ** 2 / (P1[valid] * (1.0 - P1[valid]))
    max_sigma = sigma.max()
    maxima = np.flatnonzero(np.isclose(sigma, max_sigma, rtol=1e-12, atol=1e-15))
    k_star = int(np.rint(maxima.mean()))
    return k_star, sigma

k4, sig4 = otsu_threshold(h_t4)
print('Câu 5.T4a: h =', h_t4, '-> k* =', k4)
print('σ_B²(k) =', np.round(sig4, 2))
check('Câu 5.T4a: k* = 4', lambda: k4 == 4)
k_tie, sig_tie = otsu_threshold(h_tie)
print('Câu 5.T4b: σ_B²(k) =', np.round(sig_tie, 3), '-> k* =', k_tie)
check('Câu 5.T4b: cực đại tại k = 3 và 5, k* = 4', lambda: k_tie == 4 and np.isclose(sig_tie[3], sig_tie[5]))

rice = cv2.imread(os.path.join(IMG_DIR, 'gao3.png'), cv2.IMREAD_GRAYSCALE)
k_rice, sig_rice = otsu_threshold(np.bincount(rice.ravel(), minlength=256))
t_cv, _ = cv2.threshold(rice, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f'gao3.png: Otsu tự cài đặt = {k_rice}, OpenCV = {t_cv}')
check('Otsu tự cài đặt gần OpenCV (lệch ≤ 1)', lambda: abs(k_rice - t_cv) <= 1)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(sig_rice); plt.axvline(k_rice, color='r'); plt.title('σ_B²(k) của gao3.png')
plt.subplot(1, 2, 2); plt.imshow(rice > k_rice, cmap='gray'); plt.title(f'Tách ngưỡng Otsu (T = {k_rice})'); plt.axis('off')
plt.show()

## 9. Thuật toán đẳng điệu (ISODATA)

Các bước: (1) chọn ngưỡng khởi đầu `t₀`; (2) chia điểm ảnh thành `G₁ = {g ≤ t}` và `G₂ = {g > t}`; (3) tính trung bình `μ₁`, `μ₂` có trọng số theo `h(g)`; (4) `t_mới = round((μ₁ + μ₂) / 2)`; (5) lặp lại đến khi ngưỡng không đổi.

**Yêu cầu**: viết hàm `isodata_threshold(h, t0)` trả về `(T, lịch_sử)` với `lịch_sử` là danh sách các bộ `(t, μ₁, μ₂, t_mới)`. Kiểm tra với **Câu 5.T5a** (`hA`, `t₀ = 2` → `T = 4`) rồi so sánh với Otsu trên ảnh `gao3.png` (khởi đầu `t₀` bằng giá trị xám trung bình của ảnh; lưu ý nếu chọn `t₀` lớn hơn mức xám lớn nhất của ảnh thì nhóm `G₂` rỗng).

In [ ]:
hA = np.array([6, 15, 19, 9, 4, 3, 8, 21, 12, 3])

def isodata_threshold(h, t0):
    h = np.asarray(h, dtype=np.float64)
    g = np.arange(len(h))
    t = int(t0)
    history = []
    while True:
        t = max(0, min(t, len(h) - 2))
        left_count = h[:t + 1].sum()
        right_count = h[t + 1:].sum()
        if left_count == 0 or right_count == 0:
            raise ValueError('Một trong hai nhóm của ISODATA bị rỗng')
        m1 = float((g[:t + 1] * h[:t + 1]).sum() / left_count)
        m2 = float((g[t + 1:] * h[t + 1:]).sum() / right_count)
        t_new = int(np.floor((m1 + m2) / 2.0 + 0.5))
        history.append((t, m1, m2, t_new))
        if t_new == t:
            T = t
            break
        t = t_new
    return T, history

T_A, hist_A = isodata_threshold(hA, 2)
for t, m1, m2, tn in hist_A:
    print(f't = {t}:  μ₁ = {m1:.3f},  μ₂ = {m2:.3f},  (μ₁+μ₂)/2 = {(m1 + m2) / 2:.3f}  ->  t mới = {tn}')
check('Câu 5.T5a: T = 4', lambda: T_A == 4)
T_rice, _ = isodata_threshold(np.bincount(rice.ravel(), minlength=256), int(rice.mean()))
print('gao3.png: ISODATA =', T_rice, '| Otsu =', k_rice)

## 10. Thuật toán đối xứng nền và thuật toán tam giác

**Đối xứng nền**: `maxp` là mức xám có `h` lớn nhất; `a` là mức xám nhỏ nhất sao cho phần trăm tích lũy từ `g = 0` đạt tối thiểu `p%`; ngưỡng `T = maxp − (a − maxp)`.

**Tam giác**: nối đỉnh cao nhất `(g₁, h₁)` với điểm cuối `(g₂, h₂)` của lược đồ (phía có đuôi dài, mức xám lớn nhất/nhỏ nhất còn có `h > 0`); ngưỡng là mức xám có **khoảng cách vuông góc** đến đường thẳng đó lớn nhất:
`d(g) = |(h₂ − h₁)·g − (g₂ − g₁)·h(g) + g₂h₁ − h₂g₁| / √((h₂ − h₁)² + (g₂ − g₁)²)`.

**Yêu cầu**: viết hai hàm `background_symmetry_threshold(h, p)` và `triangle_threshold(h)`. Kiểm tra với **Câu 5.T5b** (`hB`, `p = 95%` → `T = 4`) và **Câu 5.T7** (`hC` → `T = 4`). Sau đó so sánh `triangle_threshold` với `cv2.THRESH_TRIANGLE` trên ảnh `gao3.png` và đếm số hạt gạo thu được khi dùng Otsu và khi dùng tam giác (dùng `cv2.connectedComponentsWithStats`, bỏ các thành phần có diện tích < 10).

In [ ]:
hB = np.array([4, 9, 5, 3, 2, 4, 48, 17, 6, 2])
hC = np.array([3, 10, 52, 24, 9, 5, 3, 2, 1, 1])

def background_symmetry_threshold(h, p):
    h = np.asarray(h, dtype=np.float64)
    maxp = int(h.argmax())
    target = h.sum() * (p / 100.0)
    a = int(np.searchsorted(np.cumsum(h), target, side='left'))
    T = int(np.clip(2 * maxp - a, 0, len(h) - 1))
    return T

T_B = background_symmetry_threshold(hB, 95)
print('Câu 5.T5b: T =', T_B)
check('Câu 5.T5b: T = 4', lambda: T_B == 4)
print('Thử p = 88% với lược đồ [2,3,4,5,7,8,12,47,10,2] (ngân hàng Câu 4.7b):', background_symmetry_threshold([2, 3, 4, 5, 7, 8, 12, 47, 10, 2], 88), '← suy biến (T = maxp)')

In [ ]:
def triangle_threshold(h):
    h = np.asarray(h, dtype=np.float64)
    nz = np.flatnonzero(h)
    peak = int(h.argmax())
    far = nz[0] if peak - nz[0] > nz[-1] - peak else nz[-1]     # đầu mút xa đỉnh hơn
    g1, h1, g2, h2 = peak, h[peak], far, h[far]
    gs = np.arange(min(g1, g2), max(g1, g2) + 1)
    denominator = np.hypot(h2 - h1, g2 - g1)
    if denominator == 0:
        T = peak
    else:
        distances = np.abs((h2 - h1) * gs - (g2 - g1) * h[gs] + g2 * h1 - h2 * g1) / denominator
        T = int(gs[np.argmax(distances)])
    return T

T_C = triangle_threshold(hC)
print('Câu 5.T7: T =', T_C)
check('Câu 5.T7: T = 4', lambda: T_C == 4)

h_rice = np.bincount(rice.ravel(), minlength=256)
t_tri = triangle_threshold(h_rice)
t_tri_cv, _ = cv2.threshold(rice, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)
print(f'gao3.png: tam giác tự cài đặt = {t_tri}, OpenCV = {t_tri_cv}')

def count_grains(mask, min_area=10):
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    return int((stats[1:, cv2.CC_STAT_AREA] >= min_area).sum())

print('Số hạt gạo – Otsu:', count_grains(rice > k_rice), '| tam giác:', count_grains(rice > t_tri))
show([rice, rice > k_rice, rice > t_tri], ['gao3.png', f'Otsu (T = {k_rice})', f'Tam giác (T = {t_tri})'], figsize=(15, 5))

**Nhận xét**: Lược đồ của `gao3.png` có một đỉnh nền rất lớn và một đuôi dài do các hạt gạo sáng; hai lớp mất cân bằng mạnh và không tạo hai đỉnh có kích thước tương đương. Otsu tối ưu phương sai giữa lớp nên dễ bị đỉnh nền chi phối. Thuật toán tam giác đo độ lệch của phần đuôi so với đường nối từ đỉnh nền, vì vậy chọn ngưỡng ở chân đuôi và tách các hạt sáng phù hợp hơn.


## 11. Nhiều ngưỡng: Otsu nhiều lớp

Với lược đồ có nhiều hơn hai đỉnh (như Câu 5.T4b) một ngưỡng không đủ. Otsu mở rộng chọn `K − 1` ngưỡng làm cực đại phương sai giữa các lớp. Thư viện `skimage.filters.threshold_multiotsu(image, classes=3)` thực hiện việc này.

**Yêu cầu**:
1. Tạo ảnh một chiều từ lược đồ của Câu 5.T4b (`np.repeat(np.arange(10), h_tie)`) và dùng `threshold_multiotsu(..., classes=3, nbins=10)`. Kết quả mong đợi `[2, 6]` (ba lớp {0–2}, {3–6}, {7–9}).
2. Áp dụng `threshold_multiotsu` với `classes = 3` cho ảnh xám `himym.jpg` (thu nhỏ về 600×400) và hiển thị ảnh phân thành 3 lớp bằng `np.digitize`.

In [ ]:
from skimage.filters import threshold_multiotsu

tie_img = np.repeat(np.arange(10), h_tie)
th_tie = threshold_multiotsu(tie_img, classes=3, nbins=10)
print('Ngưỡng Otsu nhiều lớp của Câu 5.T4b:', th_tie)
check('Hai ngưỡng [2, 6]', lambda: list(th_tie) == [2, 6])

himym = cv2.imread(os.path.join(IMG_DIR, 'himym.jpg'), cv2.IMREAD_GRAYSCALE)
himym = cv2.resize(himym, (600, 400))

In [ ]:
th3 = threshold_multiotsu(himym, classes=3)
classes3 = np.digitize(himym, bins=th3)
print('Các ngưỡng:', th3)
show([himym, classes3], ['Ảnh xám', f'3 lớp (ngưỡng {list(th3)})'], figsize=(14, 5))

## 12. Phân vùng theo miền: tách cây tứ phân và tăng trưởng vùng

### 12.1. Tách cây tứ phân (quadtree splitting)

Một miền được coi là **thuần nhất** nếu `max − min ≤ thr`. Nếu miền không thuần nhất thì chia thành 4 miền con có kích thước bằng nhau và xét tiếp, đến khi thuần nhất hoặc chỉ còn 1 điểm ảnh.

**Yêu cầu**: viết hàm đệ quy `quadtree_split(img, thr)` trả về danh sách các miền `(hàng, cột, kích_thước)`, thứ tự duyệt 4 miền con: trên-trái, trên-phải, dưới-trái, dưới-phải. Thử với ma trận 8×8 `Q` và `thr = 5`, kiểm tra số miền; sau đó áp dụng cho ảnh `unequalized.jpg` (cắt 128×128) với `thr = 40` và vẽ ranh giới các miền.

In [ ]:
Q = np.array([[10, 11, 10, 12, 50, 52, 51, 50], [11, 10, 12, 10, 52, 50, 50, 51], [10, 12, 11, 10, 51, 50, 52, 52], [12, 10, 10, 11, 50, 51, 50, 50], [10, 10, 48, 50, 11, 10, 10, 12], [12, 11, 50, 52, 10, 12, 11, 10], [10, 10, 10, 11, 10, 10, 12, 11], [11, 12, 10, 10, 11, 10, 10, 10]])

def quadtree_split(img, thr):
    regions = []
    def rec(r, c, s):
        region = img[r:r + s, c:c + s]
        if s == 1 or int(region.max()) - int(region.min()) <= thr:
            regions.append((r, c, s))
            return
        half = s // 2
        rec(r, c, half)
        rec(r, c + half, half)
        rec(r + half, c, half)
        rec(r + half, c + half, half)
    rec(0, 0, img.shape[0])
    return regions

regs = quadtree_split(Q, 5)
print('Số miền:', len(regs)); print(regs)
check('Ma trận Q (thr = 5) được tách thành 7 miền', lambda: len(regs) == 7)
check('Các miền phủ kín ảnh, không chồng lấn', lambda: sum(s * s for _, _, s in regs) == Q.size)

unq = cv2.imread(os.path.join(IMG_DIR, 'unequalized.jpg'), cv2.IMREAD_GRAYSCALE)[:128, :128]
regs2 = quadtree_split(unq, 40)
vis = cv2.cvtColor(unq, cv2.COLOR_GRAY2BGR)
for r, c, s in regs2:
    cv2.rectangle(vis, (c, r), (c + s - 1, r + s - 1), (0, 0, 255), 1)
print('Số miền của ảnh unequalized (thr = 40):', len(regs2))
show([unq, cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)], ['Ảnh cắt 128×128', 'Ranh giới các miền'], figsize=(12, 6))

### 12.2. Tăng trưởng vùng (region growing)

Từ một **điểm hạt giống** `(r, c)`, lần lượt thêm các điểm lân cận 4 hướng có `|I(p) − mean(vùng)| ≤ thr`, trong đó `mean(vùng)` là trung bình hiện tại của vùng.

**Yêu cầu**: viết hàm `region_growing(img, seed, thr)` trả về mặt nạ nhị phân của vùng (dùng hàng đợi `collections.deque`). Áp dụng với ảnh `gao3.png`, chọn điểm hạt giống nằm trong một hạt gạo (hãy tìm bằng `np.argwhere(rice > 60)[0]`) với `thr = 25` và vẽ vùng tìm được chồng lên ảnh gốc.

In [ ]:
from collections import deque

def region_growing(img, seed, thr):
    h, w = img.shape
    mask = np.zeros((h, w), dtype=np.uint8)
    sr, sc = seed
    if not (0 <= sr < h and 0 <= sc < w):
        raise ValueError('Điểm hạt giống nằm ngoài ảnh')
    visited = np.zeros((h, w), dtype=bool)
    queue = deque([(sr, sc)])
    visited[sr, sc] = True
    mask[sr, sc] = 1
    region_sum = float(img[sr, sc])
    region_count = 1
    while queue:
        r, c = queue.popleft()
        mean_value = region_sum / region_count
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc]:
                visited[nr, nc] = True
                value = float(img[nr, nc])
                if abs(value - mean_value) <= thr:
                    mask[nr, nc] = 1
                    region_sum += value
                    region_count += 1
                    queue.append((nr, nc))
    return mask

seed_pt = tuple(int(v) for v in np.argwhere(rice > 60)[0])
region = region_growing(rice, seed_pt, 25)
print('Điểm hạt giống:', seed_pt, '| số điểm ảnh của vùng:', int(region.sum()))
check('Vùng tìm được không rỗng và nhỏ hơn 5% diện tích ảnh', lambda: 0 < region.sum() < 0.05 * rice.size)
overlay = cv2.cvtColor(rice, cv2.COLOR_GRAY2RGB)
overlay[region > 0] = (255, 0, 0)
show([rice, overlay], ['Ảnh gốc', 'Vùng tăng trưởng (màu đỏ)'], figsize=(12, 6))

## 13. Mở rộng: phân vùng bằng phân cụm K-means

Mỗi điểm ảnh được biểu diễn bằng vectơ màu `(R, G, B)`; K-means chia các vectơ thành `K` cụm và thay mỗi điểm ảnh bằng tâm cụm của nó.

**Yêu cầu**: với ảnh `images/img1.jpg`:
1. Dùng `cv2.kmeans` với `K = 2, 4, 6, 8` và hiển thị ảnh phân vùng.
2. Vẽ giá trị hàm mục tiêu (`compactness`) theo `K` từ 2 đến 12 để chọn `K` phù hợp (điểm “khuỷu tay”).

In [ ]:
img_bgr = cv2.imread(os.path.join(IMG_DIR, 'img1.jpg'))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
X = np.float32(img_rgb.reshape(-1, 3))
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1.0)
cv2.setRNGSeed(0)

def kmeans_segment(K):
    compactness, labels, centers = cv2.kmeans(
        X, K, None, criteria, 10, cv2.KMEANS_PP_CENTERS
    )
    centers = np.clip(np.rint(centers), 0, 255).astype(np.uint8)
    segmented = centers[labels.ravel()].reshape(img_rgb.shape)
    compactness = float(compactness)
    return segmented, compactness

ks = [2, 4, 6, 8]
show([img_rgb] + [kmeans_segment(k)[0] for k in ks], ['Ảnh gốc'] + [f'K = {k}' for k in ks], cmap=None, figsize=(24, 4))
objective = [kmeans_segment(k)[1] for k in range(2, 13)]
check('Hàm mục tiêu giảm khi tăng K', lambda: all(objective[i] >= objective[i + 1] - 1e-6 * objective[0] for i in range(len(objective) - 1)))
plt.figure(figsize=(7, 4)); plt.plot(range(2, 13), objective, marker='o'); plt.xlabel('K'); plt.ylabel('compactness'); plt.title('Hàm mục tiêu K-means'); plt.show()

## 14. Bài tập tổng hợp: đếm hạt gạo và thống kê diện tích

Kết hợp các kiến thức ở trên cho ảnh `gao3.png`:

1. Chọn phương pháp tách ngưỡng phù hợp (xem Mục 10), tách hạt gạo khỏi nền.
2. Dùng phép **mở** hình thái học với kernel 3×3 để khử các điểm nhiễu nhỏ (`cv2.morphologyEx`).
3. Dùng `cv2.connectedComponentsWithStats` để đếm hạt (bỏ thành phần diện tích < 30) và lấy diện tích từng hạt.
4. Tìm các biên của hạt bằng `cv2.Canny` trên mặt nạ, đếm số điểm biên (gần đúng chu vi) rồi vẽ **biểu đồ phân bố diện tích** và in diện tích trung bình, độ lệch chuẩn.

**Yêu cầu**: hoàn thành hàm `analyze_grains(gray)` trả về `(mask, areas, edge_pixels)`.

In [ ]:
def analyze_grains(gray):
    h = np.bincount(gray.ravel(), minlength=256)
    threshold = triangle_threshold(h)
    binary = np.where(gray > threshold, 255, 0).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(opened, connectivity=8)
    valid_labels = np.flatnonzero(stats[1:, cv2.CC_STAT_AREA] >= 30) + 1
    areas = stats[valid_labels, cv2.CC_STAT_AREA].astype(np.int64)
    mask = np.where(np.isin(labels, valid_labels), 255, 0).astype(np.uint8)
    edges = cv2.Canny(mask, 50, 150)
    edge_pixels = int(np.count_nonzero(edges))
    return mask, areas, edge_pixels

mask, areas, edge_pixels = analyze_grains(rice)
print(f'Số hạt: {len(areas)} | diện tích trung bình: {areas.mean():.1f} | độ lệch chuẩn: {areas.std():.1f} | số điểm biên: {edge_pixels}')
check('Đếm được từ 90 đến 110 hạt gạo', lambda: 90 <= len(areas) <= 110)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.imshow(mask, cmap='gray'); plt.title('Mặt nạ hạt gạo'); plt.axis('off')
plt.subplot(1, 2, 2); plt.hist(areas, bins=20); plt.xlabel('Diện tích (điểm ảnh)'); plt.ylabel('Số hạt'); plt.title('Phân bố diện tích hạt gạo')
plt.show()